# Database Schema Inspection

## NBA Player Impact and Lineup Optimization Dashboard

The objective of this notebook is to inspect the structure of the NBA SQLite database and identify the tables required to build an event-level analytics pipeline.

This project is designed to estimate player impact and evaluate lineup performance using play-by-play data. Before constructing possessions, stints, and lineup-level metrics, it is necessary to understand how games, events, players, and teams are stored in the database.

For this project, the database schema inspection serves four purposes:

1. Identify the tables containing play-by-play events, game metadata, team metadata, and player metadata.
2. Understand how event-level data is linked to games, players, and teams.
3. Determine whether substitutions and score progression can be reconstructed reliably.
4. Establish the data sources required to build stint-level and lineup-level datasets in later notebooks.

This step is essential because the quality of the downstream player impact model depends on the correctness of the event reconstruction pipeline.

## Objectives

By the end of this notebook, we want to answer the following questions:

- What tables are available in the SQLite database?
- Which tables are relevant for play-by-play analysis?
- Which columns define games, periods, event order, players, teams, and score?
- Which tables will be used in the next stage of the project?

The output of this notebook will be a clear schema map that guides the event cleaning and stint construction process.

In [1]:
import sqlite3
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)

DB_PATH = "../data/raw/nba.sqlite"

## Connect to the SQLite database

We begin by creating a connection to the local NBA database.This will allow us to inspect the schema and preview the available tables.

In [2]:
conn = sqlite3.connect(DB_PATH)
print("Database connection established.")

Database connection established.


## List all tables in the database

The first step is to identify the full set of tables available in the database. This gives us an overview of the information stored and helps isolate the tables most relevant for lineup analytics.

In [3]:
tables = pd.read_sql(
    """
    SELECT name
    FROM sqlite_master
    WHERE type='table'
    ORDER BY name;
    """,
    conn
)

tables

,name
0,common_player_info
1,draft_combine_stats
2,draft_history
3,game
4,game_info
5,game_summary
6,inactive_players
7,line_score
8,officials
9,other_stats


## Initial observations

From the table list, several tables appear especially relevant for this project:

- `play_by_play`: likely the main event-level table
- `game`: game-level metadata
- `game_info` and `game_summary`: supporting game context
- `player` and `common_player_info`: player identity and metadata
- `team`, `team_details`, and `team_info_common`: team metadata
- `line_score`: game scoring breakdown

At this stage, the most important next step is to inspect the schema of the `play_by_play` table and the tables that can be joined to it.

## Define helper functions for schema inspection

To keep the notebook clean and reusable, we define two utility functions:

- `get_table_columns()` to inspect the schema of a table
- `preview_table()` to inspect a small sample of rows

In [4]:
def get_table_columns(conn, table_name: str) -> pd.DataFrame:
    query = f"PRAGMA table_info({table_name});"
    return pd.read_sql(query, conn)



In [5]:
def preview_table(conn, table_name: str, limit: int = 5) -> pd.DataFrame:
    query = f"SELECT * FROM {table_name} LIMIT {limit};"
    return pd.read_sql(query, conn)



## Inspect the schema of priority tables

Rather than reviewing every table in detail, we focus first on the tables most likely to support the event reconstruction pipeline.

Priority tables for this project:

- `play_by_play`
- `game`
- `game_info`
- `game_summary`
- `player`
- `common_player_info`
- `team`
- `line_score`

In [6]:
priority_tables = [
    "play_by_play",
    "game",
    "game_info",
    "game_summary",
    "player",
    "common_player_info",
    "team",
    "line_score"
]

for table in priority_tables:
    print(f"\n--- Schema: {table} ---")
    display(get_table_columns(conn, table))


--- Schema: play_by_play ---


,cid,name,type,notnull,dflt_value,pk
0,0,game_id,TEXT,0,None,0
1,1,eventnum,INTEGER,0,None,0
2,2,eventmsgtype,INTEGER,0,None,0
3,3,eventmsgactiontype,INTEGER,0,None,0
4,4,period,INTEGER,0,None,0
5,5,wctimestring,TEXT,0,None,0
6,6,pctimestring,TEXT,0,None,0
7,7,homedescription,TEXT,0,None,0
8,8,neutraldescription,TEXT,0,None,0
9,9,visitordescription,TEXT,0,None,0



--- Schema: game ---


,cid,name,type,notnull,dflt_value,pk
0,0,season_id,TEXT,0,None,0
1,1,team_id_home,TEXT,0,None,0
2,2,team_abbreviation_home,TEXT,0,None,0
3,3,team_name_home,TEXT,0,None,0
4,4,game_id,TEXT,0,None,0
5,5,game_date,TIMESTAMP,0,None,0
6,6,matchup_home,TEXT,0,None,0
7,7,wl_home,TEXT,0,None,0
8,8,min,INTEGER,0,None,0
9,9,fgm_home,REAL,0,None,0



--- Schema: game_info ---


...

,cid,name,type,notnull,dflt_value,pk
0,0,person_id,TEXT,0,None,0
1,1,first_name,TEXT,0,None,0
2,2,last_name,TEXT,0,None,0
3,3,display_first_last,TEXT,0,None,0
4,4,display_last_comma_first,TEXT,0,None,0
5,5,display_fi_last,TEXT,0,None,0
6,6,player_slug,TEXT,0,None,0
7,7,birthdate,TIMESTAMP,0,None,0
8,8,school,TEXT,0,None,0
9,9,country,TEXT,0,None,0



--- Schema: team ---


,cid,name,type,notnull,dflt_value,pk
0,0,id,TEXT,0,None,0
1,1,full_name,TEXT,0,None,0
2,2,abbreviation,TEXT,0,None,0
3,3,nickname,TEXT,0,None,0
4,4,city,TEXT,0,None,0
5,5,state,TEXT,0,None,0
6,6,year_founded,REAL,0,None,0



--- Schema: line_score ---


,cid,name,type,notnull,dflt_value,pk
0,0,game_date_est,TIMESTAMP,0,None,0
1,1,game_sequence,INTEGER,0,None,0
2,2,game_id,TEXT,0,None,0
3,3,team_id_home,TEXT,0,None,0
4,4,team_abbreviation_home,TEXT,0,None,0
5,5,team_city_name_home,TEXT,0,None,0
6,6,team_nickname_home,TEXT,0,None,0
7,7,team_wins_losses_home,TEXT,0,None,0
8,8,pts_qtr1_home,TEXT,0,None,0
9,9,pts_qtr2_home,TEXT,0,None,0


##  Preview the main tables

After identifying the columns in each priority table, we inspect a small sample of rows to understand how the information is stored in practice.

This step helps validate:
- identifier formats
- naming conventions
- score representation
- player/team linkage
- event text fields

In [7]:
for table in priority_tables:
    print(f"\n--- Preview: {table} ---")
    display(preview_table(conn, table, limit=3))


--- Preview: play_by_play ---


,game_id,eventnum,eventmsgtype,eventmsgactiontype,period,wctimestring,pctimestring,homedescription,neutraldescription,visitordescription,score,scoremargin,person1type,player1_id,player1_name,player1_team_id,player1_team_city,player1_team_nickname,player1_team_abbreviation,person2type,player2_id,player2_name,player2_team_id,player2_team_city,player2_team_nickname,player2_team_abbreviation,person3type,player3_id,player3_name,player3_team_id,player3_team_city,player3_team_nickname,player3_team_abbreviation,video_available_flag
0,0029600012,0,12,0,1,14:43 PM,12:00,None,Start of 1st Period (14:43 PM EST),None,None,None,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0
1,0029600012,2,10,0,1,14:50 PM,12:00,Jump Ball O'Neal vs. Kleine: Tip to Cassell,None,None,None,None,4.0,406,Shaquille O'Neal,1610612747.0,Los Angeles,Lakers,LAL,5.0,170,Joe Kleine,1610612756.0,Phoenix,Suns,PHX,5.0,208,Sam Cassell,1610612756.0,Phoenix,Suns,PHX,0
2,0029600012,3,2,1,1,14:51 PM,11:45,None,None,MISS Cassell 15' Jump Shot,None,None,5.0,208,Sam Cassell,1610612756.0,Phoenix,Suns,PHX,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0



--- Preview: game ---


,season_id,team_id_home,team_abbreviation_home,team_name_home,game_id,game_date,matchup_home,wl_home,min,fgm_home,fga_home,fg_pct_home,fg3m_home,fg3a_home,fg3_pct_home,ftm_home,fta_home,ft_pct_home,oreb_home,dreb_home,reb_home,ast_home,stl_home,blk_home,tov_home,pf_home,pts_home,plus_minus_home,video_available_home,team_id_away,team_abbreviation_away,team_name_away,matchup_away,wl_away,fgm_away,fga_away,fg_pct_away,fg3m_away,fg3a_away,fg3_pct_away,ftm_away,fta_away,ft_pct_away,oreb_away,dreb_away,reb_away,ast_away,stl_away,blk_away,tov_away,pf_away,pts_away,plus_minus_away,video_available_away,season_type
0,21946,1610610035,HUS,Toronto Huskies,0024600001,1946-11-01 00:00:00,HUS vs. NYK,L,0,25.0,NaN,NaN,None,None,None,16.0,29.0,0.552,None,None,None,None,None,None,None,NaN,66.0,-2,0,1610612752,NYK,New York Knicks,NYK @ HUS,W,24.0,NaN,NaN,None,None,None,20.0,26.0,0.769,None,None,None,None,None,None,None,NaN,68.0,2,0,Regular Season
1,21946,1610610034,BOM,St. Louis Bombers,0024600003,1946-11-02 00:00:00,BOM vs. PIT,W,0,20.0,59.0,0.339,None,None,None,16.0,NaN,NaN,None,None,None,None,None,None,None,21.0,56.0,5,0,1610610031,PIT,Pittsburgh Ironmen,PIT @ BOM,L,16.0,72.0,0.222,None,None,None,19.0,NaN,NaN,None,None,None,None,None,None,None,25.0,51.0,-5,0,Regular Season
2,21946,1610610032,PRO,Providence Steamrollers,0024600002,1946-11-02 00:00:00,PRO vs. BOS,W,0,21.0,NaN,NaN,None,None,None,17.0,NaN,NaN,None,None,None,None,None,None,None,NaN,59.0,6,0,1610612738,BOS,Boston Celtics,BOS @ PRO,L,21.0,NaN,NaN,None,None,None,11.0,NaN,NaN,None,None,None,None,None,None,None,NaN,53.0,-6,0,Regular Season



--- Preview: game_info ---


...

,person_id,first_name,last_name,display_first_last,display_last_comma_first,display_fi_last,player_slug,birthdate,school,country,last_affiliation,height,weight,season_exp,jersey,position,rosterstatus,games_played_current_season_flag,team_id,team_name,team_abbreviation,team_code,team_city,playercode,from_year,to_year,dleague_flag,nba_flag,games_played_flag,draft_year,draft_round,draft_number,greatest_75_flag
0,76001,Alaa,Abdelnaby,Alaa Abdelnaby,"Abdelnaby, Alaa",A. Abdelnaby,alaa-abdelnaby,1968-06-24 00:00:00,Duke,USA,Duke/USA,6-10,240,5.0,30,Forward,Inactive,N,1610612757,Trail Blazers,POR,blazers,Portland,HISTADD_alaa_abdelnaby,1990.0,1994.0,N,Y,Y,1990,1,25,N
1,76002,Zaid,Abdul-Aziz,Zaid Abdul-Aziz,"Abdul-Aziz, Zaid",Z. Abdul-Aziz,zaid-abdul-aziz,1946-04-07 00:00:00,Iowa State,USA,Iowa State/USA,6-9,235,10.0,54,Center,Inactive,N,1610612745,Rockets,HOU,rockets,Houston,HISTADD_zaid_abdul-aziz,1968.0,1977.0,N,Y,Y,1968,1,5,N
2,76003,Kareem,Abdul-Jabbar,Kareem Abdul-Jabbar,"Abdul-Jabbar, Kareem",K. Abdul-Jabbar,kareem-abdul-jabbar,1947-04-16 00:00:00,UCLA,USA,UCLA/USA,7-2,225,20.0,33,Center,Inactive,N,1610612747,Lakers,LAL,lakers,Los Angeles,HISTADD_kareem_abdul-jabbar,1969.0,1988.0,N,Y,Y,1969,1,1,Y



--- Preview: team ---


,id,full_name,abbreviation,nickname,city,state,year_founded
0,1610612737,Atlanta Hawks,ATL,Hawks,Atlanta,Georgia,1949.0
1,1610612738,Boston Celtics,BOS,Celtics,Boston,Massachusetts,1946.0
2,1610612739,Cleveland Cavaliers,CLE,Cavaliers,Cleveland,Ohio,1970.0



--- Preview: line_score ---


,game_date_est,game_sequence,game_id,team_id_home,team_abbreviation_home,team_city_name_home,team_nickname_home,team_wins_losses_home,pts_qtr1_home,pts_qtr2_home,pts_qtr3_home,pts_qtr4_home,pts_ot1_home,pts_ot2_home,pts_ot3_home,pts_ot4_home,pts_ot5_home,pts_ot6_home,pts_ot7_home,pts_ot8_home,pts_ot9_home,pts_ot10_home,pts_home,team_id_away,team_abbreviation_away,team_city_name_away,team_nickname_away,team_wins_losses_away,pts_qtr1_away,pts_qtr2_away,pts_qtr3_away,pts_qtr4_away,pts_ot1_away,pts_ot2_away,pts_ot3_away,pts_ot4_away,pts_ot5_away,pts_ot6_away,pts_ot7_away,pts_ot8_away,pts_ot9_away,pts_ot10_away,pts_away
0,1946-11-01 00:00:00,None,0024600001,1610610035,HUS,Toronto,Huskies,-,None,None,None,None,18.0,None,None,None,None,None,None,None,None,None,66.0,1610612752,NYK,New York,Knicks,-,NaN,None,None,NaN,24.0,None,None,None,None,None,None,None,None,None,68.0
1,1946-11-02 00:00:00,None,0024600003,1610610034,BOM,St. Louis,Bombers,-,16,16,18,6,NaN,None,None,None,None,None,None,None,None,None,56.0,1610610031,PIT,Pittsburgh,Ironmen,-,5.0,15,17,14.0,NaN,None,None,None,None,None,None,None,None,None,51.0
2,1946-11-02 00:00:00,None,0024600002,1610612738,BOS,Boston,Celtics,-,10.0,16,14,13,NaN,None,None,None,None,None,None,None,None,None,53.0,1610610032,PRO,Providence,Steamrollers,-,NaN,12,18,15.0,NaN,None,None,None,None,None,None,None,None,None,59.0


## Focus on the play-by-play table

The `play_by_play` table is the most important source for this project.

To build stint-level data later, we need to verify whether this table contains enough information to reconstruct:

- game sequence
- period and game clock
- event ordering
- player participation
- team attribution
- score changes
- substitutions

We now inspect the play-by-play schema more closely.

In [8]:
pbp_columns = get_table_columns(conn, "play_by_play")
pbp_columns

,cid,name,type,notnull,dflt_value,pk
0,0,game_id,TEXT,0,None,0
1,1,eventnum,INTEGER,0,None,0
2,2,eventmsgtype,INTEGER,0,None,0
3,3,eventmsgactiontype,INTEGER,0,None,0
4,4,period,INTEGER,0,None,0
5,5,wctimestring,TEXT,0,None,0
6,6,pctimestring,TEXT,0,None,0
7,7,homedescription,TEXT,0,None,0
8,8,neutraldescription,TEXT,0,None,0
9,9,visitordescription,TEXT,0,None,0


In [9]:
pbp_sample = preview_table(conn, "play_by_play", limit=10)
pbp_sample

,game_id,eventnum,eventmsgtype,eventmsgactiontype,period,wctimestring,pctimestring,homedescription,neutraldescription,visitordescription,score,scoremargin,person1type,player1_id,player1_name,player1_team_id,player1_team_city,player1_team_nickname,player1_team_abbreviation,person2type,player2_id,player2_name,player2_team_id,player2_team_city,player2_team_nickname,player2_team_abbreviation,person3type,player3_id,player3_name,player3_team_id,player3_team_city,player3_team_nickname,player3_team_abbreviation,video_available_flag
0,0029600012,0,12,0,1,14:43 PM,12:00,None,Start of 1st Period (14:43 PM EST),None,None,None,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0
1,0029600012,2,10,0,1,14:50 PM,12:00,Jump Ball O'Neal vs. Kleine: Tip to Cassell,None,None,None,None,4.0,406,Shaquille O'Neal,1610612747.0,Los Angeles,Lakers,LAL,5.0,170,Joe Kleine,1610612756.0,Phoenix,Suns,PHX,5.0,208,Sam Cassell,1610612756.0,Phoenix,Suns,PHX,0
2,0029600012,3,2,1,1,14:51 PM,11:45,None,None,MISS Cassell 15' Jump Shot,None,None,5.0,208,Sam Cassell,1610612756.0,Phoenix,Suns,PHX,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0
3,0029600012,4,4,0,1,14:51 PM,11:43,O'Neal REBOUND (Off:0 Def:1),None,None,None,None,4.0,406,Shaquille O'Neal,1610612747.0,Los Angeles,Lakers,LAL,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0
4,0029600012,5,2,1,1,14:51 PM,11:29,MISS Ceballos 26' 3PT Jump Shot,None,None,None,None,4.0,76,Cedric Ceballos,1610612747.0,Los Angeles,Lakers,LAL,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0
5,0029600012,6,4,0,1,14:51 PM,11:27,None,None,Cassell REBOUND (Off:0 Def:1),None,None,5.0,208,Sam Cassell,1610612756.0,Phoenix,Suns,PHX,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0
6,0029600012,7,6,1,1,14:51 PM,11:14,Van Exel P.FOUL (P1.T1),None,None,None,None,4.0,89,Nick Van Exel,1610612747.0,Los Angeles,Lakers,LAL,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0
7,0029600012,8,5,1,1,14:52 PM,11:08,None,None,Cassell Bad Pass Turnover (P1.T1),None,None,5.0,208,Sam Cassell,1610612756.0,Phoenix,Suns,PHX,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0
8,0029600012,9,2,5,1,14:52 PM,10:49,MISS Ceballos 1' Layup,None,Horry BLOCK (1 BLK),None,None,4.0,76,Cedric Ceballos,1610612747.0,Los Angeles,Lakers,LAL,0.0,0,None,None,None,None,None,5.0,109,Robert Horry,1610612756.0,Phoenix,Suns,PHX,0
9,0029600012,10,4,0,1,14:52 PM,10:49,LAKERS Rebound,None,None,None,None,2.0,1610612747,None,None,None,None,None,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0


## Interpretation of the play-by-play table

At this point, the goal is not only to inspect the table, but to determine whether it can support lineup reconstruction.

The most important fields to validate are:

- **game identifier**: required to isolate a single game
- **event sequence / event number**: required to order game events correctly
- **period and clock**: required to preserve game chronology
- **player identifiers**: required to link events to players
- **team identifiers**: required to assign events to teams
- **score fields**: required to calculate point differential
- **event description fields**: required to classify substitutions, scoring plays, rebounds, fouls, and turnovers

If these fields are present and consistent, the database is suitable for building stint-level and lineup-level analytics.

In [10]:
schema_summary = pd.DataFrame({
    "table_name": [
        "play_by_play", "game", "game_info", "game_summary",
        "player", "common_player_info", "team", "line_score"
    ],
    "role_in_project": [
        "Event-level game actions and score progression",
        "Core game identifiers and game metadata",
        "Additional game context",
        "Game summary and supporting metadata",
        "Player identifiers and names",
        "Extended player information",
        "Team identifiers and names",
        "Score breakdown by game/period"
    ],
    "priority_level": [
        "High", "High", "Medium", "Medium",
        "High", "Medium", "High", "Medium"
    ]
})

schema_summary

,table_name,role_in_project,priority_level
0,play_by_play,Event-level game actions and score progression,High
1,game,Core game identifiers and game metadata,High
2,game_info,Additional game context,Medium
3,game_summary,Game summary and supporting metadata,Medium
4,player,Player identifiers and names,High
5,common_player_info,Extended player information,Medium
6,team,Team identifiers and names,High
7,line_score,Score breakdown by game/period,Medium


## Selected tables for the analytics pipeline

Based on the schema inspection, the project will rely primarily on the following tables:

- `play_by_play` for event reconstruction
- `game` for game-level identifiers and context
- `player` for player mapping
- `team` for team mapping

Additional supporting tables such as `game_info`, `game_summary`, and `line_score` may be used for validation and enrichment.

## Key findings

This notebook establishes the structural foundation of the project by identifying the database tables relevant to event-level NBA analysis.

The database contains dedicated tables for:
- game metadata
- play-by-play events
- player metadata
- team metadata
- line scores and supporting context


## Next step

The next notebook will focus on event-level exploration of the `play_by_play` table.

Main goals for Notebook 02:
- identify key event types
- classify scoring and substitution events
- understand period and clock formatting
- validate whether lineup stints can be reconstructed reliably